# 02 — Random Forest Training

Dieses Notebook trainiert **Random-Forest-Regressionsmodelle** auf allen 5 Datenvarianten.

## Was ist ein Random Forest?
Ein Random Forest ist ein **Ensemble aus unabhaengigen Entscheidungsbaeumen** (Bagging):
- Jeder Baum wird auf einer zufaelligen Teilmenge der Daten trainiert (Bootstrap-Sampling)
- Jeder Split verwendet nur eine zufaellige Teilmenge der Features (Feature-Bagging)
- Die Vorhersage ist der **Mittelwert** aller Baum-Vorhersagen

## Unterschied zu XGBoost
| | Random Forest | XGBoost |
|---|---|---|
| Bauweise | Parallel (unabhaengig) | Sequenziell (jeder Baum korrigiert Fehler) |
| Early Stopping | Nicht moeglich | Ja (ueber Val-Set) |
| GPU | Nicht verfuegbar (sklearn) | Ja (CUDA) |
| Laufzeit | Langsam bei grossen Daten | Schnell mit GPU |

**Laufzeit**: ca. 3 Stunden (CPU, D4 ist sehr gross)

In [ ]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

SCRIPT_START = time.time()

In [ ]:
SCRIPT_DIR = Path(".").resolve().parent
DATA_DIR = SCRIPT_DIR / "data"
MODEL_DIR = SCRIPT_DIR / "models"
OUTPUT_DIR = SCRIPT_DIR / "ergebnisse"
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "travel_time"
VARIANTEN = ["D1_single", "D2_multi4", "D3_mittel6", "D4_gross10", "D5_fremd"]

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — ignoriert Nullwerte im Nenner."""
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

## Hyperparameter

| Parameter | Wert | Erklaerung |
|-----------|------|------------|
| n_estimators | 300 | 300 unabhaengige Baeume (mehr = stabiler, aber langsamer) |
| max_depth | 20 | Tiefere Baeume als XGBoost, da kein Boosting-Kaskaden-Effekt |
| min_samples_leaf | 10 | Mindestens 10 Samples pro Blatt (Regularisierung) |
| max_features | 0.8 | 80% der Features pro Baum (Feature-Bagging) |
| n_jobs | -1 | Alle CPU-Kerne nutzen (Baeume sind unabhaengig) |

In [ ]:
results = []

for variant in VARIANTEN:
    print(f"\n{'='*60}")
    print(f"  Random Forest | {variant}")
    print(f"{'='*60}")

    t0 = time.time()
    d = DATA_DIR / variant
    train = pd.read_csv(d / "train.csv")
    test = pd.read_csv(d / "test.csv")
    features = [c for c in train.columns if c != TARGET]

    X_train, y_train = train[features].values, train[TARGET].values
    X_test, y_test = test[features].values, test[TARGET].values

    print(f"  Train: {len(train):,}  Test: {len(test):,}  Features: {len(features)}")

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=10,
        max_features=0.8,
        n_jobs=-1,
        random_state=42,
        verbose=1,
    )
    model.fit(X_train, y_train)

    # Evaluation nur auf Test-Set (kein Early Stopping)
    y_pred_test = model.predict(X_test)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = root_mean_squared_error(y_test, y_pred_test)
    mape_test = mape(y_test, y_pred_test)

    elapsed = time.time() - t0
    print(f"\n  Test MAE: {mae_test:.2f}s  RMSE: {rmse_test:.2f}s  MAPE: {mape_test:.1f}%")
    print(f"  Dauer: {elapsed:.1f}s")

    model_path = MODEL_DIR / f"rf_{variant}.joblib"
    joblib.dump(model, model_path)

    results.append({
        "experiment": f"RF_{variant}",
        "modell": "RandomForest",
        "daten": variant,
        "train_n": len(train),
        "mae_test": round(mae_test, 2),
        "rmse_test": round(rmse_test, 2),
        "mape_test": round(mape_test, 1),
        "zeit_s": round(elapsed, 1),
    })

## Zusammenfassung

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "rf_results.csv", index=False)
results_df[["experiment", "mae_test", "rmse_test", "mape_test", "zeit_s"]]

In [ ]:
print(f"Gesamtlaufzeit: {time.time() - SCRIPT_START:.1f}s")